# 04 — Quantum Machine Learning: Variational Quantum Classifier (PennyLane, simulator only)
**CodeCraft · Early Metabolic Risk Detection · NHANES 2017–2018**

> ⚠️ **Research/educational use only. Not a diagnosis or medical advice.**
> The data are **cross-sectional**: results describe classification *within this survey cycle*, **not** prediction of future disease.
> All metrics are **unweighted** (survey weights are not available). **Simulator only — no real quantum hardware is used, and no quantum advantage is claimed.** No claim is made about which model family (classical ML, deep learning, quantum ML) is better; only real runs from this notebook are reported.

**Task:** binary classification, positive class = *Elevated Risk* (HbA1c 5.7 % to < 6.5 %, label 1) vs *Normal* (< 5.7 %, label 0).

## How to run
- **Needed in Drive** (`MyDrive/CodeCraft/`): `train.csv`, `validation.csv`, `test.csv`, `split_manifest.json`, and **`results/shared_feature_set.json` created by `02_ml_model.ipynb`** (run order 02 → 03 → 04 → 05).
- *Runtime → Run all* on a fresh **free Colab CPU** runtime (no GPU needed). The first cell installs PennyLane; if Colab asks to restart the runtime after the install, restart and run again.
- **Expected runtime:** the design target is roughly 10–15 minutes per seed (5 seeds ≈ 1 hour in total; the per-epoch time is printed so you can check early). Each seed has a **20-minute time budget**. Training uses a **stratified subsample of the train set** (default 1500 rows) to keep simulation affordable; evaluation uses the **full** validation and test sets.
- **Resumable:** each seed's result is saved immediately; after a Colab disconnect just run all again — completed seeds are skipped. Everything is written under `MyDrive/CodeCraft/results/qml/` and `results/predictions/` (prediction CSVs are private working files, never part of an exported report).

## Section 1 — Setup, helpers, data loading, TC-06 test, shared feature set, angle-encoding preprocessing
Install PennyLane (only if missing), mount Drive, define the common helper code (identical in every notebook), load the fixed splits (never re-split), run TC-06, learn pruning on **train only**, load the shared feature set (its `split_hash` must match), then build the angle-encoding preprocessing: standard preprocessing → `MinMaxScaler((0, π))` fitted on **train** → val/test **clipped** to [0, π].

In [ ]:
import importlib, subprocess, sys
# PennyLane is not preinstalled on Colab: equivalent to `pip install -q pennylane`, but only if missing
for mod, pkg in [('pennylane', 'pennylane'), ('torch', 'torch'), ('sklearn', 'scikit-learn'), ('pandas', 'pandas'),
                 ('numpy', 'numpy'), ('matplotlib', 'matplotlib'), ('joblib', 'joblib'), ('scipy', 'scipy')]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
import pennylane as qml
print('PennyLane version:', qml.__version__)

In [ ]:
# ===== COMMON HELPER CODE (identical in every notebook) =====
import numpy as np, pandas as pd, json, os, time, hashlib, resource, platform, sys
from pathlib import Path
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/CodeCraft')
SEED = 42
DATASET_VERSION = 'nhanes-2017-2018-v1'
WARNING_TEXT = 'Research/educational use only. Not a diagnosis or medical advice.'

def file_sha256(paths):
    h = hashlib.sha256()
    for p in paths: h.update(Path(p).read_bytes())
    return h.hexdigest()

def write_json_atomic(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, default=str)); os.replace(tmp, path)

def peak_memory_mb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0

def compute_metrics(y_true, y_prob, thr):
    y_true = np.asarray(y_true).astype(int); y_pred = (np.asarray(y_prob) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    div = lambda a, b: float(a / b) if b > 0 else None
    prec, sens, spec = div(tp, tp + fp), div(tp, tp + fn), div(tn, tn + fp)
    f1 = div(2 * prec * sens, prec + sens) if (prec is not None and sens is not None) else None
    auc = float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else None
    return {'accuracy': float((tp + tn) / len(y_true)), 'precision': prec, 'sensitivity': sens,
            'specificity': spec, 'f1': f1, 'roc_auc': auc,
            'confusion': [[int(tn), int(fp)], [int(fn), int(tp)]]}  # [[TN,FP],[FN,TP]]

def pick_threshold(y_val, p_val, min_spec=0.70):
    fpr, tpr, thr = roc_curve(y_val, p_val)
    ok = np.where(((1 - fpr) >= min_spec) & np.isfinite(thr))[0]
    if len(ok) == 0: return 0.5, False
    return float(thr[ok[np.argmax(tpr[ok])]]), True

def bootstrap_ci(y_true, y_prob, thr, n=1000, seed=42):
    rng = np.random.default_rng(seed); y_true = np.asarray(y_true); y_prob = np.asarray(y_prob)
    keys = ['accuracy', 'precision', 'sensitivity', 'specificity', 'f1', 'roc_auc']
    out = {k: [] for k in keys}
    for _ in range(n):
        i = rng.integers(0, len(y_true), len(y_true)); m = compute_metrics(y_true[i], y_prob[i], thr)
        for k in keys:
            if m[k] is not None: out[k].append(m[k])
    return {k: ([float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] if v else None) for k, v in out.items()}

def evaluate_run(y_val, p_val, y_test, p_test):
    thr, found = pick_threshold(y_val, p_val)
    return {'threshold': thr, 'threshold_found': found,
            'metrics_at_threshold': compute_metrics(y_test, p_test, thr),
            'metrics_at_0.5': compute_metrics(y_test, p_test, 0.5),
            'ci95': bootstrap_ci(y_test, p_test, thr),
            'test_n': int(len(y_test)), 'test_prevalence': float(np.mean(y_test))}

In [ ]:
import random, tempfile, shutil, numbers
import matplotlib, scipy, sklearn, joblib, torch
import matplotlib.pyplot as plt

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
set_seed(SEED)

BACKEND = 'QML simulator: default.qubit'
RESULTS_DIR = BASE_DIR / 'results'
QML_DIR, PRED_DIR, FIG_DIR = RESULTS_DIR / 'qml', RESULTS_DIR / 'predictions', RESULTS_DIR / 'qml' / 'figures'
for d in [RESULTS_DIR, QML_DIR, PRED_DIR, FIG_DIR, RESULTS_DIR / 'ml', RESULTS_DIR / 'dl']:
    d.mkdir(parents=True, exist_ok=True)

VERSIONS = {'python': sys.version.split()[0], 'numpy': np.__version__, 'pandas': pd.__version__, 'scikit-learn': sklearn.__version__,
            'scipy': scipy.__version__, 'matplotlib': matplotlib.__version__, 'joblib': joblib.__version__,
            'torch': torch.__version__, 'pennylane': qml.__version__}
print('Library versions:', json.dumps(VERSIONS))
print(f'Backend: {BACKEND} (classical simulation on CPU; no quantum hardware) | base seed {SEED} | {platform.platform()}')
assert BASE_DIR.exists(), 'BASE_DIR not found - is Google Drive mounted and the CodeCraft folder present?'
print('[VERIFIED] Drive mounted; results folders ready under', RESULTS_DIR)

In [ ]:
# ---- Load the three fixed splits (never re-split) ----
TARGET, POSITIVE, NEGATIVE = 'risk_label', 'Elevated Risk', 'Normal'
FORBIDDEN = ['SEQN', 'LBXGH', 'WTINT2YR', 'WTMEC2YR', 'SDMVPSU', 'SDMVSTRA']
PATHS = {s: BASE_DIR / f'{s}.csv' for s in ['train', 'validation', 'test']}
missing_files = [str(p) for p in PATHS.values() if not p.exists()]
if missing_files:
    raise FileNotFoundError('Missing split file(s) in Drive: ' + ', '.join(missing_files))

train_df, val_df, test_df = (pd.read_csv(PATHS[s]) for s in ['train', 'validation', 'test'])
for name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    assert TARGET in df.columns, f'{name}: column {TARGET!r} missing'
    assert not df.columns.duplicated().any(), f'{name}: duplicated column names'
    bad = [c for c in df.columns if c.upper() in FORBIDDEN]
    assert not bad, f'{name}: forbidden columns present: {bad}'
    assert df[TARGET].notna().all(), f'{name}: missing labels'
    assert set(df[TARGET].unique()) <= {NEGATIVE, POSITIVE}, f'{name}: unexpected labels {df[TARGET].unique()}'
    assert df[TARGET].nunique() == 2, f'{name}: both classes needed'
assert list(train_df.columns) == list(val_df.columns) == list(test_df.columns), 'Column mismatch between splits'

y_train, y_val, y_test = (((d[TARGET] == POSITIVE).astype(int)).to_numpy() for d in (train_df, val_df, test_df))
X_train, X_val, X_test = (d.drop(columns=[TARGET]) for d in (train_df, val_df, test_df))

split_hash = file_sha256([PATHS['train'], PATHS['validation'], PATHS['test']])
manifest_path = BASE_DIR / 'split_manifest.json'
print('split_manifest.json present:', manifest_path.exists())
print('split_hash:', split_hash)
for n, X, y in [('train', X_train, y_train), ('validation', X_val, y_val), ('test', X_test, y_test)]:
    print(f'{n:<10} rows={len(X):>6}  predictors={X.shape[1]}  prevalence(Elevated Risk)={y.mean():.3f}')
print('\nPredictor columns, dtypes and missingness in TRAIN (inspected at runtime):')
print(pd.DataFrame({'dtype': X_train.dtypes.astype(str), 'n_unique': X_train.nunique(dropna=True),
                    'pct_missing': (X_train.isna().mean() * 100).round(1)}).to_string())
print('[VERIFIED] splits loaded; no forbidden columns; identical columns across splits; both classes in every split')

In [ ]:
# ---- TC-06: compute_metrics vs scikit-learn on a fixed fixture ----
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
fx_y = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0])
fx_p = np.array([0.10, 0.40, 0.35, 0.80, 0.65, 0.90, 0.20, 0.70, 0.55, 0.30, 0.45, 0.60])
m = compute_metrics(fx_y, fx_p, 0.5); fx_pred = (fx_p >= 0.5).astype(int)
expected = {'accuracy': accuracy_score(fx_y, fx_pred), 'precision': precision_score(fx_y, fx_pred),
            'sensitivity': recall_score(fx_y, fx_pred), 'specificity': recall_score(fx_y, fx_pred, pos_label=0),
            'f1': f1_score(fx_y, fx_pred), 'roc_auc': roc_auc_score(fx_y, fx_p)}
for k, v in expected.items():
    assert round(m[k], 4) == round(float(v), 4), f'TC-06 mismatch for {k}: {m[k]} vs {v}'
# Rule 4: metrics that cannot be computed must be None (never 0)
m0 = compute_metrics([0, 0, 0], [0.1, 0.2, 0.3], 0.5)
assert m0['sensitivity'] is None and m0['precision'] is None and m0['f1'] is None and m0['roc_auc'] is None
print('[PASSED] TC-06: compute_metrics matches scikit-learn to 4 decimals; undefined metrics return None')

In [ ]:
# ---- Feature-column pruning: learned from TRAIN only ----
ADMIN_EXACT = {'BMDSTATS', 'RIDSTATR', 'SDDSRVYR', 'RIDEXMON'}

def learn_pruning(Xtr):
    dropped = {}
    for c in Xtr.columns:
        s = Xtr[c]; miss = s.isna().mean()
        if (c.startswith('BMI') and c != 'BMXBMI') or c in ADMIN_EXACT:
            dropped[c] = 'administrative / comment-code column'
        elif miss > 0.5:
            dropped[c] = f'>50% missing in train ({miss:.1%})'
        elif s.nunique(dropna=True) <= 1:
            dropped[c] = 'constant in train'
    return dropped

def classify_column(s):
    # Integer-coded columns with <=10 unique non-null values are categorical (even if stored as float)
    if (not pd.api.types.is_numeric_dtype(s)) or pd.api.types.is_bool_dtype(s):
        return 'categorical'
    v = s.dropna().to_numpy(dtype=float)
    if v.size and np.all(np.isfinite(v)) and np.all(v == np.round(v)) and len(np.unique(v)) <= 10:
        return 'categorical'
    return 'numeric'

DROPPED = learn_pruning(X_train)
KEPT_COLS = [c for c in X_train.columns if c not in DROPPED]
COL_TYPE = {c: classify_column(X_train[c]) for c in KEPT_COLS}
NUMERIC_COLS = [c for c in KEPT_COLS if COL_TYPE[c] == 'numeric']
CAT_COLS = [c for c in KEPT_COLS if COL_TYPE[c] == 'categorical']

print('Dropped columns (reason):')
print('\n'.join(f'  - {c}: {r}' for c, r in DROPPED.items()) if DROPPED else '  (none)')
print(f'\nKept {len(KEPT_COLS)} of {X_train.shape[1]} predictors: {len(NUMERIC_COLS)} numeric, {len(CAT_COLS)} categorical')
print(pd.DataFrame({'type': pd.Series(COL_TYPE), 'dtype': X_train[KEPT_COLS].dtypes.astype(str),
                    'n_unique': X_train[KEPT_COLS].nunique(dropna=True),
                    'pct_missing': (X_train[KEPT_COLS].isna().mean() * 100).round(1)}).to_string())
assert len(KEPT_COLS) > 0 and TARGET not in KEPT_COLS
assert not [c for c in KEPT_COLS if c.upper() in FORBIDDEN]
assert set(NUMERIC_COLS) | set(CAT_COLS) == set(KEPT_COLS) and not set(NUMERIC_COLS) & set(CAT_COLS)
print('[VERIFIED] pruning learned on train only; numeric/categorical partition is complete')

In [ ]:
# ---- Standard preprocessing (identical in all notebooks) ----
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def make_preprocessor(numeric_cols, categorical_cols):
    transformers = []
    if numeric_cols:
        transformers.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                                              ('sc', StandardScaler())]), list(numeric_cols)))
    if categorical_cols:
        transformers.append(('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                                              ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False,
                                                                   drop='if_binary'))]), list(categorical_cols)))
    return ColumnTransformer(transformers, remainder='drop', verbose_feature_names_out=False)

# sanity check on all kept columns: fit on TRAIN only, transform val/test
_pp = make_preprocessor(NUMERIC_COLS, CAT_COLS).fit(X_train[KEPT_COLS])
for n, X in [('train', X_train), ('validation', X_val), ('test', X_test)]:
    Z = _pp.transform(X[KEPT_COLS])
    assert np.isfinite(Z).all() and Z.shape[1] == len(_pp.get_feature_names_out())
print(f'[VERIFIED] preprocessing fitted on train only; {Z.shape[1]} transformed columns for the full pruned set')

In [ ]:
# ---- Load the shared reduced feature set created by 02_ml_model.ipynb ----
SHARED_PATH = RESULTS_DIR / 'shared_feature_set.json'
if not SHARED_PATH.exists():
    raise FileNotFoundError(f'Run 02_ml_model.ipynb first: {SHARED_PATH} was not found.')
shared = json.loads(SHARED_PATH.read_text())
need = ['selected_features', 'numeric_features', 'categorical_features', 'transformed_dim', 'mi_ranking', 'split_hash', 'seed']
if [k for k in need if k not in shared]:
    raise ValueError('shared_feature_set.json is incomplete. Run 02_ml_model.ipynb first.')
if shared['split_hash'] != split_hash:
    raise RuntimeError('shared_feature_set.json was created for different split files (split_hash mismatch). Run 02_ml_model.ipynb first.')

MAIN_FEATURES = list(shared['selected_features'])
assert set(MAIN_FEATURES) <= set(KEPT_COLS), 'shared features not among the pruned train columns'
assert not [c for c in MAIN_FEATURES if c.upper() in FORBIDDEN]
assert set(shared['numeric_features']) == {c for c in MAIN_FEATURES if COL_TYPE[c] == 'numeric'}
assert set(shared['categorical_features']) == {c for c in MAIN_FEATURES if COL_TYPE[c] == 'categorical'}
_dim = len(make_preprocessor([c for c in MAIN_FEATURES if c in NUMERIC_COLS], [c for c in MAIN_FEATURES if c in CAT_COLS])
           .fit(X_train[MAIN_FEATURES]).get_feature_names_out())
assert _dim == shared['transformed_dim'] <= 8, f'transformed_dim mismatch: {_dim} vs {shared["transformed_dim"]}'
print('Shared features:', MAIN_FEATURES)
print(f'[VERIFIED] shared feature set loaded; split_hash matches; transformed_dim = {_dim} (<= 8)')

In [ ]:
# ---- Angle-encoding preprocessing: standard preprocessing -> MinMaxScaler(0, pi) fit on TRAIN -> clip val/test to [0, pi] ----
from sklearn.preprocessing import MinMaxScaler
N_QUBITS = int(shared['transformed_dim'])
assert 2 <= N_QUBITS <= 8, f'n_qubits must be between 2 and 8, got {N_QUBITS}'

class AngleEncoder:
    def __init__(self, numeric, categorical):
        self.numeric, self.categorical = list(numeric), list(categorical)
    def fit(self, X):                                           # X = TRAIN rows only
        self.cols_ = self.numeric + self.categorical
        self.pre_ = make_preprocessor(self.numeric, self.categorical).fit(X[self.cols_])
        self.scaler_ = MinMaxScaler(feature_range=(0, np.pi)).fit(self.pre_.transform(X[self.cols_]))
        return self
    def raw_scaled(self, X): return self.scaler_.transform(self.pre_.transform(X[self.cols_]))
    def transform(self, X): return np.clip(self.raw_scaled(X), 0.0, np.pi)
    def n_features(self): return int(len(self.pre_.get_feature_names_out()))

_enc = AngleEncoder([c for c in MAIN_FEATURES if c in NUMERIC_COLS], [c for c in MAIN_FEATURES if c in CAT_COLS]).fit(X_train)
assert _enc.n_features() == N_QUBITS
for n, X in [('train', X_train), ('validation', X_val), ('test', X_test)]:
    raw = _enc.raw_scaled(X); Z = _enc.transform(X)
    assert Z.min() >= 0 and Z.max() <= np.pi + 1e-12 and np.isfinite(Z).all()
    print(f'{n:<10} shape={Z.shape}  entries clipped to [0, pi]: {np.mean((raw < 0) | (raw > np.pi)):.2%}')
print(f'[VERIFIED] n_qubits = transformed_dim = {N_QUBITS} (<= 8); scaler fitted on train only; all encoded values in [0, pi]')

## Section 2 — The variational circuit
- **Device:** `default.qubit`, `wires = n_qubits`, **analytic** (`shots=None`), differentiated by backpropagation through the torch interface.
- **Encoding:** `qml.AngleEmbedding(features, rotation='Y')` — one feature per qubit as an RY angle (features scaled to [0, π]).
- **Ansatz:** `qml.StronglyEntanglingLayers(weights)` with `n_layers = 2`, weights shape `(n_layers, n_qubits, 3)`.
- **Measurement:** `⟨Z⟩` on qubit 0, a value `z ∈ [-1, 1]`.
- **Classical read-out (documented):** probability `p = sigmoid(a · z + b)` with two **learnable classical scalars** `a` (initialised 1) and `b` (initialised 0), trained together with the circuit weights. The model's parameter count is `n_layers · n_qubits · 3 + 2`.
- **Speed:** parameter broadcasting — a whole mini-batch (or evaluation batch) goes through the circuit in one call.

In [ ]:
# QML_CORE: all PennyLane / PyTorch specific code lives in this cell
def build_circuit(n_qubits):
    dev = qml.device('default.qubit', wires=n_qubits, shots=None)       # analytic simulation
    @qml.qnode(dev, interface='torch', diff_method='backprop')
    def circuit(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation='Y')
        qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
        return qml.expval(qml.PauliZ(0))
    return dev, circuit

def init_params(n_layers, n_qubits, seed):
    g = torch.Generator(); g.manual_seed(int(seed))
    w = (torch.rand(n_layers, n_qubits, 3, generator=g, dtype=torch.float64) * 2 * np.pi).requires_grad_(True)
    a = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)      # classical read-out scale
    b = torch.tensor(0.0, dtype=torch.float64, requires_grad=True)      # classical read-out bias
    return {'w': w, 'a': a, 'b': b}

def qml_logits(params, X):
    # X: float64 tensor (batch, n_qubits) -> logit = a * <Z0> + b  (sigmoid(logit) = probability of Elevated Risk)
    z = CIRCUIT(X, params['w']).reshape(-1).to(torch.float64)
    return params['a'] * z + params['b']

def train_qml(Ztr, ytr, cfg, seed, should_stop):
    # Class-weighted BCE (pos_weight = n_neg/n_pos), Adam, mini-batches. Only TRAIN rows are used here.
    set_seed(seed)
    params = init_params(cfg['n_layers'], Ztr.shape[1], seed)
    n_pos = int(np.sum(ytr)); n_neg = int(len(ytr) - n_pos); assert n_pos > 0 and n_neg > 0
    pw = n_neg / n_pos
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw], dtype=torch.float64))
    opt = torch.optim.Adam([params['w'], params['a'], params['b']], lr=cfg['lr'])
    X = torch.from_numpy(np.ascontiguousarray(Ztr, dtype=np.float64)); y = torch.from_numpy(np.asarray(ytr, dtype=np.float64))
    g = torch.Generator(); g.manual_seed(int(seed))
    bs = cfg['batch_size']; steps_per_epoch = int(np.ceil(len(X) / bs))
    ep_loss, step_loss, stopped, reason, full_epochs = [], [], False, None, 0
    t0 = time.time()
    for epoch in range(cfg['epochs']):
        perm = torch.randperm(len(X), generator=g); run, cnt, te = 0.0, 0, time.time()
        for i in range(0, len(perm), bs):
            if should_stop():
                stopped, reason = True, f'time budget reached during epoch {epoch + 1}/{cfg["epochs"]}'; break
            idx = perm[i:i + bs]
            opt.zero_grad()
            loss = loss_fn(qml_logits(params, X[idx]), y[idx]); loss.backward(); opt.step()
            step_loss.append(float(loss.item())); run += float(loss.item()) * len(idx); cnt += len(idx)
        if cnt > 0: ep_loss.append(run / cnt)
        if not stopped: full_epochs += 1
        print(f'    seed {seed} epoch {epoch + 1:>2}/{cfg["epochs"]}  train loss {ep_loss[-1] if ep_loss else float("nan"):.4f}  ({time.time() - te:.1f}s, total {time.time() - t0:.0f}s)')
        if stopped: break
    hist = {'train_loss': ep_loss, 'step_loss': step_loss, 'steps_per_epoch': steps_per_epoch, 'epochs_completed': full_epochs,
            'stopped_early': stopped, 'stop_reason': reason, 'pos_weight': float(pw), 'train_sec': round(time.time() - t0, 1),
            'n_parameters': int(params['w'].numel() + 2)}
    return {k: v.detach().clone() for k, v in params.items()}, hist

def predict_qml(params, Z, batch_size=256):
    out = []
    with torch.no_grad():
        for i in range(0, len(Z), batch_size):
            Xb = torch.from_numpy(np.ascontiguousarray(Z[i:i + batch_size], dtype=np.float64))
            out.append(torch.sigmoid(qml_logits(params, Xb)).numpy())
    return np.concatenate(out).astype(np.float64)

def save_params(params, path):
    write_json_atomic({'w': params['w'].tolist(), 'a': float(params['a']), 'b': float(params['b'])}, path)

def load_params(path):
    d = json.loads(Path(path).read_text())
    return {'w': torch.tensor(d['w'], dtype=torch.float64), 'a': torch.tensor(d['a'], dtype=torch.float64), 'b': torch.tensor(d['b'], dtype=torch.float64)}

def describe_circuit(n_qubits, n_layers):
    x = torch.linspace(0.3, 2.8, n_qubits, dtype=torch.float64); w = init_params(n_layers, n_qubits, SEED)['w'].detach()
    info = {'diagram': qml.draw(CIRCUIT, decimals=2)(x, w), 'depth': None, 'depth_source': 'unavailable in this PennyLane version'}
    try:
        sp = qml.specs(CIRCUIT)(x, w)
        get = lambda o, k: o[k] if isinstance(o, dict) and k in o else getattr(o, k, None)
        res = get(sp, 'resources'); depth = getattr(res, 'depth', None) if res is not None else None
        if depth is None: depth = get(sp, 'depth')
        if depth is not None: info.update(depth=int(depth), depth_source='qml.specs')
    except Exception as e:
        info['depth_source'] = f'qml.specs failed: {type(e).__name__}'
    return info
print('[VERIFIED] circuit, training and prediction functions defined')

In [ ]:
N_LAYERS = 2
DEV, CIRCUIT = build_circuit(N_QUBITS)
CIRCUIT_INFO = describe_circuit(N_QUBITS, N_LAYERS)
N_QUANTUM_WEIGHTS = N_LAYERS * N_QUBITS * 3
N_PARAMETERS = N_QUANTUM_WEIGHTS + 2
print(f'Circuit for {N_QUBITS} qubits, {N_LAYERS} StronglyEntanglingLayers (example input angles, random weights):\n')
print(CIRCUIT_INFO['diagram'])
print(f"\nweights shape = ({N_LAYERS}, {N_QUBITS}, 3) -> {N_QUANTUM_WEIGHTS} circuit parameters + 2 read-out scalars (a, b) = {N_PARAMETERS} trainable parameters")
print(f"circuit depth = {CIRCUIT_INFO['depth']} (source: {CIRCUIT_INFO['depth_source']})")
write_json_atomic({k: v for k, v in CIRCUIT_INFO.items()}, QML_DIR / 'circuit_info.json')
(QML_DIR / 'circuit_diagram.txt').write_text(CIRCUIT_INFO['diagram'])
assert N_PARAMETERS == N_LAYERS * N_QUBITS * 3 + 2
print('[VERIFIED] circuit built and drawn')

## Section 3 — Training setup and adapter
Class-weighted BCE on the read-out logit (`pos_weight = n_neg / n_pos` of the training rows actually used), Adam (**lr = 0.05, fixed — not tuned**), mini-batches of 64, **15 epochs, no early stopping** (final weights are used; validation data is used only to choose the decision threshold). Because simulation is slow, training uses a **seeded stratified subsample of the TRAIN set** (`SUBSAMPLE_N`, default 1500; the subsample is drawn per run seed). Scalers/encoders are fitted on the **full** train matrix; **validation and test are always evaluated in full**, in batches.

**Time budget:** each seed has `TIME_BUDGET_MIN` minutes of training. If exceeded, `ON_BUDGET_EXCEEDED = 'fail'` (default) stops gracefully, writes a `failed` result with a clear message (no test evaluation, earlier completed results untouched); `'complete_labelled'` instead evaluates the partly trained model and labels the run as stopped early in its config.

In [ ]:
# ---- Configurable settings ----
SUBSAMPLE_N = 1500            # stratified training subsample size (>= train size means: use all train rows)
EPOCHS, BATCH_SIZE, LR = 15, 64, 0.05
TIME_BUDGET_MIN = 20          # per-seed training budget (minutes)
ON_BUDGET_EXCEEDED = 'fail'   # 'fail' or 'complete_labelled'
SEEDS = [42, 43, 44, 45, 46]
assert ON_BUDGET_EXCEEDED in ('fail', 'complete_labelled')

MODEL_CFG = {'n_layers': N_LAYERS, 'lr': LR, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'subsample_n': SUBSAMPLE_N, 'eval_batch_size': 256}
MODEL_RT = json.loads(json.dumps(MODEL_CFG))
print('MODEL_CFG (results depend on these):', MODEL_CFG)
print(f'Time budget: {TIME_BUDGET_MIN} min per seed, on exceed: {ON_BUDGET_EXCEEDED}')

from sklearn.model_selection import train_test_split
def stratified_subsample(y, n, seed):
    y = np.asarray(y)
    if n >= len(y): return np.arange(len(y))
    return np.sort(train_test_split(np.arange(len(y)), train_size=int(n), stratify=y, random_state=int(seed))[0])

class TimeBudgetExceeded(RuntimeError): pass

class QMLAdapter:
    # fit(X, y) / predict_proba(X) -> positive-class probability / describe() -> dict
    def __init__(self, cfg, feature_cols, seed, time_budget_sec=None):
        self.cfg, self.feature_cols, self.seed, self.time_budget_sec = dict(cfg), list(feature_cols), int(seed), time_budget_sec
        self.encoder_ = self.params_ = self.history_ = self.subsample_idx_ = None
    def fit(self, X, y):
        y = np.asarray(y).astype(int)
        self.encoder_ = AngleEncoder([c for c in self.feature_cols if c in NUMERIC_COLS], [c for c in self.feature_cols if c in CAT_COLS]).fit(X)   # full TRAIN
        Z = self.encoder_.transform(X)
        self.subsample_idx_ = stratified_subsample(y, self.cfg['subsample_n'], self.seed)
        ys = y[self.subsample_idx_]
        print(f'  seed {self.seed}: training subsample = {len(ys)} of {len(y)} train rows (stratified, seeded); '
              f'class counts [Normal, Elevated] = {np.bincount(ys, minlength=2).tolist()}, prevalence {ys.mean():.3f} (train {y.mean():.3f})')
        t_start = time.time()
        stop = (lambda: False) if self.time_budget_sec is None else (lambda: time.time() - t_start > self.time_budget_sec)
        self.params_, self.history_ = train_qml(Z[self.subsample_idx_], ys, self.cfg, self.seed, stop)
        self.history_['subsample_rows'] = int(len(ys)); self.history_['train_rows_total'] = int(len(y))
        return self
    def predict_proba(self, X):
        return predict_qml(self.params_, self.encoder_.transform(X), self.cfg['eval_batch_size'])
    def describe(self):
        return {'family': 'qml', 'model_name': 'VQC', 'framework': 'PennyLane default.qubit (simulator)', 'hyperparameters': self.cfg, 'seed': self.seed,
                'input_features': self.feature_cols, 'n_qubits': self.encoder_.n_features() if self.encoder_ else None,
                'n_parameters': None if self.history_ is None else self.history_['n_parameters'], 'positive_class': POSITIVE}
    def save_artifacts(self, directory, tag):
        directory = Path(directory)
        save_params(self.params_, directory / f'weights_{tag}.json')
        tmp = directory / f'preprocessing_{tag}.joblib.tmp'
        joblib.dump({'preprocessor': self.encoder_.pre_, 'scaler': self.encoder_.scaler_, 'columns': self.encoder_.cols_}, tmp)
        os.replace(tmp, directory / f'preprocessing_{tag}.joblib')
        write_json_atomic({**self.history_, 'config': self.cfg, 'seed': self.seed, 'feature_set': self.feature_cols}, directory / f'history_{tag}.json')
    def load_artifacts(self, directory, tag):
        directory = Path(directory); d = joblib.load(directory / f'preprocessing_{tag}.joblib')
        enc = AngleEncoder([c for c in self.feature_cols if c in NUMERIC_COLS], [c for c in self.feature_cols if c in CAT_COLS])
        enc.pre_, enc.scaler_, enc.cols_ = d['preprocessor'], d['scaler'], d['columns']
        self.encoder_, self.params_ = enc, load_params(directory / f'weights_{tag}.json')
        self.history_ = json.loads((directory / f'history_{tag}.json').read_text()); return self

fmt = lambda v, nd=3: 'n/a' if v is None else f'{v:.{nd}f}'
_iso_now = lambda: pd.Timestamp.now(tz='UTC').isoformat()
print('[VERIFIED] settings and QMLAdapter defined')

In [ ]:
# ---- Run-result helpers (contract: results/<family>/run_<experiment>_seed<seed>.json) ----
def _common_fields(experiment, family, seed):
    return {'runId': f'{family}_{experiment}_seed{seed}', 'family': family, 'experiment': experiment, 'seed': int(seed),
            'timestamp': _iso_now(), 'dataset_version': DATASET_VERSION, 'split_hash': split_hash,
            'positive_class': POSITIVE, 'versions': VERSIONS, 'backend': BACKEND}

def _write_result(payload, results_dir, family, experiment, seed, preds=None):
    # Never overwrite an existing COMPLETED result with a FAILED one.
    path = Path(results_dir) / family / f'run_{experiment}_seed{seed}.json'
    if payload['status'] == 'failed' and path.exists():
        try: old = json.loads(path.read_text())
        except Exception: old = {}
        if old.get('status') == 'completed':
            print(f'[KEPT] existing completed result preserved: {path.name}'); return path, False
    if preds is not None:
        pp = Path(results_dir) / 'predictions' / f'{family}_{experiment}_seed{seed}.csv'
        pp.parent.mkdir(parents=True, exist_ok=True)
        tmp = pp.with_suffix('.csv.tmp'); preds.to_csv(tmp, index=False); os.replace(tmp, pp)
    write_json_atomic(payload, path)
    return path, True

def _failed_payload(experiment, seed, exc, t0, family='qml'):
    msg = f'{type(exc).__name__}: {" ".join(str(exc).split())[:300]}'      # readable, no stack trace
    return {**_common_fields(experiment, family, seed), 'model_name': None, 'status': 'failed', 'error': msg,
            'feature_set': None, 'n_transformed_features': None, 'config': {},
            'runtime_sec': round(time.time() - t0, 2), 'peak_memory_mb': round(peak_memory_mb(), 1)}

def safe_run(experiment, seed, builder, results_dir=None, family='qml'):
    # builder() -> dict(model_name, feature_set, n_transformed_features, config, evaluation, predictions)
    results_dir = Path(results_dir or RESULTS_DIR); t0 = time.time()
    try:
        out = builder()
        payload = {**_common_fields(experiment, family, seed), 'model_name': out['model_name'], 'status': 'completed', 'error': None,
                   'feature_set': out['feature_set'], 'n_transformed_features': out['n_transformed_features'], 'config': out['config'],
                   'runtime_sec': round(time.time() - t0, 2), 'peak_memory_mb': round(peak_memory_mb(), 1), **out['evaluation']}
        preds = out['predictions']
    except Exception as e:
        payload, preds = _failed_payload(experiment, seed, e, t0, family), None
        print(f'[FAILED] run {experiment} seed {seed}: {payload["error"]}')
    _write_result(payload, results_dir, family, experiment, seed, preds)
    return payload

RUNS = {}
def make_qml_builder(experiment, seed, feats):
    def _builder():
        ad = QMLAdapter(MODEL_CFG, feats, seed, time_budget_sec=TIME_BUDGET_MIN * 60).fit(X_train, y_train)
        h = ad.history_
        if h['stopped_early'] and ON_BUDGET_EXCEEDED == 'fail':
            raise TimeBudgetExceeded(f"Time budget of {TIME_BUDGET_MIN} min exceeded ({h['stop_reason']}); stopped before validation/test evaluation. "
                                     f"Re-run to retry, or raise TIME_BUDGET_MIN / lower SUBSAMPLE_N or EPOCHS.")
        p_val, p_test = ad.predict_proba(X_val), ad.predict_proba(X_test)      # full validation / test sets, in batches
        ev = evaluate_run(y_val, p_val, y_test, p_test)                          # the ONE test scoring of this final model
        RUNS[(experiment, seed)] = {'adapter': ad}
        cfg = {'model': 'Variational quantum classifier (PennyLane)', 'frozen_config': MODEL_RT, 'n_qubits': N_QUBITS, 'n_layers': N_LAYERS,
               'weights_shape': [N_LAYERS, N_QUBITS, 3], 'n_parameters': h['n_parameters'], 'n_circuit_parameters': N_QUANTUM_WEIGHTS,
               'circuit_depth': CIRCUIT_INFO['depth'], 'circuit_depth_source': CIRCUIT_INFO['depth_source'],
               'encoding': 'AngleEmbedding(rotation=Y) on features scaled by MinMaxScaler(0, pi) fit on train; val/test clipped to [0, pi]',
               'ansatz': 'StronglyEntanglingLayers', 'observable': 'PauliZ(0)', 'readout': 'p = sigmoid(a * <Z0> + b), a and b learnable scalars',
               'loss': 'BCEWithLogits with pos_weight = n_neg/n_pos on the training rows used', 'pos_weight': h['pos_weight'],
               'optimizer': f'Adam(lr={LR}) (lr fixed, not tuned)', 'epochs': EPOCHS, 'epochs_completed': h['epochs_completed'], 'batch_size': BATCH_SIZE,
               'early_stopping': 'none (final weights used; validation only for the threshold)',
               'subsample_size': h['subsample_rows'], 'train_rows_total': h['train_rows_total'], 'subsample': 'stratified, seeded with the run seed',
               'simulator': 'default.qubit (analytic, shots=None, backprop, torch interface, parameter broadcasting)', 'train_sec': h['train_sec'],
               'time_budget_min': TIME_BUDGET_MIN, 'on_budget_exceeded': ON_BUDGET_EXCEEDED, 'stopped_early_time_budget': bool(h['stopped_early']),
               'note': ('STOPPED EARLY by the time budget: partly trained model (' + str(h['stop_reason']) + ')') if h['stopped_early'] else None,
               'threshold_rule': 'max sensitivity on validation subject to specificity >= 0.70; applied once to test', 'metrics_weighting': 'unweighted',
               'feature_set_type': 'shared reduced feature set (results/shared_feature_set.json)'}
        return {'model_name': 'VQC', 'feature_set': list(feats), 'n_transformed_features': N_QUBITS, 'config': cfg, 'evaluation': ev,
                'predictions': pd.DataFrame({'row_pos': np.arange(len(y_test)), 'y_true': y_test, 'y_prob': p_test})}
    return _builder
print('[VERIFIED] failure-safe runner and builder defined')

## Section 4 — Five seeds (42–46)
For **each seed**: train on the seeded stratified subsample, choose the threshold on the **full validation set**, evaluate on the **full test set once**, and **immediately** save `run_main_seed<seed>.json` + the predictions CSV. Seeds with a completed result (same split, features and config) are skipped, so the cell can simply be re-run after a disconnect. Seed 42 additionally saves its weights, preprocessing and training history in `results/qml/`. Peak memory is the process-wide peak so far.

In [ ]:
def artifacts_ok(tag): return all((QML_DIR / f).exists() for f in [f'weights_{tag}.json', f'preprocessing_{tag}.joblib', f'history_{tag}.json'])

def existing_completed(experiment, seed, feats, need_artifacts=False):
    p = QML_DIR / f'run_{experiment}_seed{seed}.json'
    if not p.exists(): return None
    try: r = json.loads(p.read_text())
    except Exception: return None
    ok = (r.get('status') == 'completed' and r.get('split_hash') == split_hash and r.get('feature_set') == list(feats)
          and r.get('config', {}).get('frozen_config') == MODEL_RT and (PRED_DIR / f'qml_{experiment}_seed{seed}.csv').exists())
    if ok and need_artifacts and not artifacts_ok(f'{experiment}_seed{seed}'): ok = False
    return r if ok else None

main_runs, T_ALL = {}, time.time()
for sd in SEEDS:
    r = existing_completed('main', sd, MAIN_FEATURES, need_artifacts=(sd == 42))
    if r is not None:
        print(f'[SKIPPED] seed {sd}: completed result already exists'); main_runs[sd] = r; continue
    print(f'--- seed {sd} ---')
    r = safe_run('main', sd, make_qml_builder('main', sd, MAIN_FEATURES))
    if r['status'] == 'completed':                              # saved immediately by safe_run; extras for seed 42 follow
        if sd == 42: RUNS[('main', 42)]['adapter'].save_artifacts(QML_DIR, 'main_seed42')
        main_runs[sd] = r; m = r['metrics_at_threshold']
        print(f"[DONE] seed {sd}: thr={r['threshold']:.4f} sens={fmt(m['sensitivity'])} spec={fmt(m['specificity'])} AUC={fmt(m['roc_auc'])} "
              f"| train {r['config']['train_sec']}s, total {r['runtime_sec']}s" + (' | STOPPED EARLY (time budget)' if r['config']['stopped_early_time_budget'] else ''))

tbl = []
for sd in SEEDS:
    r = main_runs.get(sd)
    if r is None: tbl.append({'seed': sd, 'status': 'failed/missing (re-run to retry)'}); continue
    m, m5 = r['metrics_at_threshold'], r['metrics_at_0.5']
    tbl.append({'seed': sd, 'thr (val)': round(r['threshold'], 4), 'sens': fmt(m['sensitivity']), 'spec': fmt(m['specificity']), 'ROC-AUC': fmt(m['roc_auc']),
                'acc': fmt(m['accuracy']), 'sens@0.5': fmt(m5['sensitivity']), 'spec@0.5': fmt(m5['specificity']), 'runtime s': r['runtime_sec'],
                'peak MB': r['peak_memory_mb'], 'early stop': 'YES' if r['config'].get('stopped_early_time_budget') else ''})
print('\nPer-seed TEST results (unweighted):'); print(pd.DataFrame(tbl).to_string(index=False))
assert len(main_runs) >= 1, 'no seed completed - see the [FAILED] messages above'
if len(main_runs) < len(SEEDS): print(f'[WARNING] only {len(main_runs)}/{len(SEEDS)} seeds completed - re-run this cell to retry the rest')
else: print('[VERIFIED] all 5 seeds completed and saved (run_main_seed42..46.json + predictions CSVs)')

## Section 5 — Summary across seeds and plots for seed 42
Mean ± sample SD (ddof = 1) over the completed seeds; threshold chosen on validation per seed; metrics at threshold 0.5 reported separately. **The SD reflects seed variation only** — here the training-subsample draw, weight initialisation and batch order — on one fixed split and one fixed full test set; it is not a confidence interval for generalisation (per-run bootstrap CIs are in each JSON). Plots use line styles, markers and printed numbers in addition to colour.

In [ ]:
loaded = {}
for sd in SEEDS:
    p = QML_DIR / f'run_main_seed{sd}.json'
    if p.exists():
        r = json.loads(p.read_text())
        if r.get('status') == 'completed' and r.get('split_hash') == split_hash: loaded[sd] = r
assert len(loaded) >= 1
assert all(r['feature_set'] == MAIN_FEATURES and r['config']['frozen_config'] == MODEL_RT for r in loaded.values()), 'runs use different features/configs'
KEYS = ['sensitivity', 'specificity', 'precision', 'f1', 'accuracy', 'roc_auc']
def agg(vals):
    v = [x for x in vals if x is not None]
    if not v: return {'mean': None, 'sd': None, 'n': 0, 'values': vals}
    return {'mean': float(np.mean(v)), 'sd': float(np.std(v, ddof=1)) if len(v) > 1 else None, 'n': len(v), 'values': vals}
ms = lambda a: 'n/a' if a['mean'] is None else (f"{a['mean']:.3f} +/- {a['sd']:.3f}" if a['sd'] is not None else f"{a['mean']:.3f} (n=1)")
ss = sorted(loaded)
at_thr = {k: agg([loaded[s]['metrics_at_threshold'][k] for s in ss]) for k in KEYS}
at_05 = {k: agg([loaded[s]['metrics_at_0.5'][k] for s in ss]) for k in KEYS}
thr_agg = agg([loaded[s]['threshold'] for s in ss])
early = [s for s in ss if loaded[s]['config'].get('stopped_early_time_budget')]
summary = {'family': 'qml', 'experiment': 'main', 'model_name': 'VQC', 'seeds': ss, 'n_seeds': len(ss), 'seeds_stopped_early_by_time_budget': early,
           'frozen_config': MODEL_RT, 'n_qubits': N_QUBITS, 'n_parameters': N_PARAMETERS, 'feature_set': MAIN_FEATURES, 'split_hash': split_hash,
           'dataset_version': DATASET_VERSION, 'positive_class': POSITIVE, 'backend': BACKEND, 'threshold': thr_agg,
           'metrics_at_threshold': at_thr, 'metrics_at_0.5': at_05, 'runtime_sec': agg([loaded[s]['runtime_sec'] for s in ss]),
           'sd_note': 'SD (ddof=1) reflects seed variation only (training-subsample draw, weight init, batch order) on one fixed split and test set; metrics are unweighted.',
           'limits_note': 'Simulator only, <= 8 qubits, trained on a stratified train subsample; no quantum advantage claimed; not comparable to a hardware run.',
           'warning': WARNING_TEXT, 'timestamp': _iso_now()}
write_json_atomic(summary, QML_DIR / 'summary_main.json')
print(f"Mean +/- SD over seeds {ss} (TEST, unweighted); threshold chosen on validation per seed")
print(pd.DataFrame({'metric': KEYS, 'at val-chosen threshold': [ms(at_thr[k]) for k in KEYS], 'at threshold 0.5': [ms(at_05[k]) for k in KEYS]}).to_string(index=False))
print('mean validation-chosen threshold:', ms(thr_agg))
if early: print('[NOTE] seeds stopped early by the time budget (partly trained):', early)
print('NOTE: SD reflects seed variation only (fixed split, fixed test set); it is not a confidence interval for generalisation.')
assert (QML_DIR / 'summary_main.json').exists()
print('[VERIFIED] summary_main.json saved')

In [ ]:
from matplotlib.lines import Line2D
r42 = json.loads((QML_DIR / 'run_main_seed42.json').read_text()); assert r42['status'] == 'completed', 'seed 42 not completed'
hist42 = json.loads((QML_DIR / 'history_main_seed42.json').read_text())
pred42 = pd.read_csv(PRED_DIR / 'qml_main_seed42.csv'); th, t5 = r42['metrics_at_threshold'], r42['metrics_at_0.5']
assert (pred42.y_true.to_numpy() == y_test).all()

ep = np.arange(1, len(hist42['train_loss']) + 1); st = hist42['step_loss']; spe = hist42['steps_per_epoch']
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot((np.arange(len(st)) + 1) / spe, st, linestyle=':', linewidth=1, color='gray', label='Mini-batch loss (each step)')
ax.plot(ep, hist42['train_loss'], linestyle='-', marker='o', markersize=4, color='black', label='Mean training loss per epoch')
ax.set_xlabel('Epoch'); ax.set_ylabel('Class-weighted BCE loss'); ax.set_title(f"Seed 42: VQC training loss ({hist42['subsample_rows']} training rows)"); ax.legend(loc='upper right')
plt.tight_layout(); plt.savefig(FIG_DIR / 'loss_curve_seed42.png', dpi=150); plt.show()

fpr, tpr, _ = roc_curve(pred42.y_true, pred42.y_prob)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, linestyle='-', linewidth=2, color='black', label=f"VQC seed 42 (ROC-AUC = {fmt(th['roc_auc'])})")
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance (AUC = 0.5)')
ax.axvline(0.30, linestyle=':', color='gray', label='Specificity = 0.70 (FPR = 0.30)')
if th['sensitivity'] is not None and th['specificity'] is not None:
    ax.plot(1 - th['specificity'], th['sensitivity'], marker='o', markersize=10, markerfacecolor='white', markeredgecolor='black', linestyle='None', label=f"Val-chosen threshold {r42['threshold']:.3f}")
if t5['sensitivity'] is not None and t5['specificity'] is not None:
    ax.plot(1 - t5['specificity'], t5['sensitivity'], marker='s', markersize=9, markerfacecolor='black', markeredgecolor='black', linestyle='None', label='Threshold 0.5')
ax.set_xlabel('False positive rate (1 - specificity)'); ax.set_ylabel('True positive rate (sensitivity)')
ax.set_title('ROC curve - TEST set (seed 42, unweighted)'); ax.legend(loc='lower right'); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
plt.tight_layout(); plt.savefig(FIG_DIR / 'roc_seed42.png', dpi=150); plt.show()

cm = np.array(th['confusion']); tot = cm.sum()
fig, ax = plt.subplots(figsize=(5.5, 5)); im = ax.imshow(cm, cmap='Greys', vmin=0, vmax=cm.max() * 1.25)
lab = [['TN', 'FP'], ['FN', 'TP']]
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{lab[i][j]}\n{cm[i, j]}\n({cm[i, j] / tot:.1%})', ha='center', va='center', color='white' if cm[i, j] > cm.max() * 0.6 else 'black', fontsize=12)
ax.set_xticks([0, 1]); ax.set_xticklabels(['Normal (0)', 'Elevated Risk (1)']); ax.set_yticks([0, 1]); ax.set_yticklabels(['Normal (0)', 'Elevated Risk (1)'])
ax.set_xlabel('Predicted class'); ax.set_ylabel('Actual class'); ax.set_title(f"Confusion matrix - TEST, seed 42, threshold {r42['threshold']:.3f}")
plt.colorbar(im, ax=ax).set_label('Count')
ax.legend(handles=[Line2D([], [], linestyle='', label=t) for t in ['TN true negative', 'FP false positive', 'FN false negative', 'TP true positive']], loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)
plt.tight_layout(); plt.savefig(FIG_DIR / 'confusion_seed42.png', dpi=150); plt.show()
print('[VERIFIED] loss curve, ROC and confusion matrix for seed 42 created and saved under results/qml/figures')

## Section 6 — Honest note on limits
- **Simulator cost:** a classical computer simulates the circuit exactly (`default.qubit`, analytic). Cost doubles with every added qubit, which is why the model is capped at **≤ 8 qubits** and why training uses a **stratified subsample** of the training set (default 1500 rows, 15 epochs); validation and test are evaluated in full.
- **Few qubits = few inputs:** one feature (or one-hot column) per qubit means only the shared reduced feature set (≤ 8 transformed columns) can be used. All three model families use this same set in the headline comparison.
- **Small, fixed model:** 2 entangling layers, a single measured qubit, learning rate fixed at 0.05 and **not tuned**; no early stopping. Results depend on these choices and on the random weight initialisation.
- **No quantum advantage is claimed.** An exact simulation of a small circuit is just a (slow) classical computation; nothing here shows, or is intended to show, any speed or accuracy benefit from quantum hardware.
- **Not comparable to hardware:** there is no shot noise, no gate noise and no device connectivity constraints. Results from a real quantum device would differ and cannot be inferred from these runs.
- **Seed SD ≠ generalisation uncertainty** (see Section 5), and the train subsample means this model saw less training data than the classical and deep-learning models; any difference between families must be read with that in mind.

## Section 7 — Single-record demo, failure-isolation test, final summary
`predict_single_record` loads the **saved seed-42 weights and preprocessing** from `results/qml/`, validates every field **before** the model is called (invalid input returns a field-level error list and the circuit is never run). Output is a research score, not a diagnosis.

In [ ]:
DEMO = QMLAdapter(MODEL_CFG, MAIN_FEATURES, 42).load_artifacts(QML_DIR, 'main_seed42')
assert np.allclose(DEMO.predict_proba(X_test), pred42.y_prob.to_numpy(), atol=1e-6), 'reloaded weights/preprocessing do not reproduce saved seed-42 predictions'
print('[VERIFIED] saved weights + preprocessing reproduce the seed-42 test probabilities')

SINGLE_NUM_RANGE = {c: (float(X_train[c].min()), float(X_train[c].max())) for c in MAIN_FEATURES if c in NUMERIC_COLS}
SINGLE_CAT_LEVELS = {c: set(X_train[c].dropna().unique().tolist()) for c in MAIN_FEATURES if c in CAT_COLS}
MAIN_THRESHOLD = r42['threshold']; _CALLS = {'n': 0}

def _score_record(df):
    _CALLS['n'] += 1
    return float(DEMO.predict_proba(df)[0])

def predict_single_record(feature_dict):
    # Required fields = shared feature set. Numeric: finite number inside observed train range +/- 50% of its span.
    # Categorical: must be a category code seen in train. Missing/None rejected. Model is NOT called if any error.
    errors = []
    if not isinstance(feature_dict, dict):
        return {'valid': False, 'errors': [{'field': '<input>', 'error': 'input must be a dict of field -> value'}]}
    for k in feature_dict:
        if str(k).upper() in FORBIDDEN: errors.append({'field': k, 'error': 'field is not allowed as a predictor'})
    for c in MAIN_FEATURES:
        if c not in feature_dict or feature_dict[c] is None:
            errors.append({'field': c, 'error': 'required field is missing'}); continue
        v = feature_dict[c]
        if c in SINGLE_NUM_RANGE:
            if isinstance(v, bool) or not isinstance(v, numbers.Real) or not np.isfinite(v):
                errors.append({'field': c, 'error': f'must be a finite number, got {v!r}'}); continue
            lo, hi = SINGLE_NUM_RANGE[c]; span = hi - lo
            if not (lo - 0.5 * span <= v <= hi + 0.5 * span):
                errors.append({'field': c, 'error': f'{v} outside plausible range [{lo - 0.5 * span:.4g}, {hi + 0.5 * span:.4g}]'})
        else:
            try: ok = v in SINGLE_CAT_LEVELS[c]
            except TypeError: ok = False
            if not ok: errors.append({'field': c, 'error': f'{v!r} is not a known category; allowed: {sorted(SINGLE_CAT_LEVELS[c])}'})
    if errors: return {'valid': False, 'errors': errors}
    score = _score_record(pd.DataFrame([{c: feature_dict[c] for c in MAIN_FEATURES}]))
    return {'model': 'VQC (PennyLane simulator, seed 42)', 'score': score, 'label': POSITIVE if score >= MAIN_THRESHOLD else NEGATIVE,
            'threshold': MAIN_THRESHOLD, 'warning': WARNING_TEXT}

complete = X_val[MAIN_FEATURES].notna().all(axis=1); assert complete.any()
valid_rec = X_val.loc[complete.idxmax(), MAIN_FEATURES].to_dict()
print('Valid example input:  ', valid_rec)
b = _CALLS['n']; res_ok = predict_single_record(valid_rec); print('Result:', res_ok)
assert _CALLS['n'] == b + 1 and set(res_ok) == {'model', 'score', 'label', 'threshold', 'warning'} and 0 <= res_ok['score'] <= 1

bad_rec = dict(valid_rec); num_demo = next((c for c in MAIN_FEATURES if c in NUMERIC_COLS), None)
if num_demo: bad_rec[num_demo] = 'abc'
drop_demo = [c for c in MAIN_FEATURES if c != num_demo][:1] or [MAIN_FEATURES[0]]; bad_rec.pop(drop_demo[0], None)
print('\nInvalid example input:', bad_rec)
b = _CALLS['n']; res_bad = predict_single_record(bad_rec); print('Result:', json.dumps(res_bad, indent=1, default=str))
assert res_bad['valid'] is False and len(res_bad['errors']) >= 1 and _CALLS['n'] == b
print('[PASSED] valid record scored; invalid record rejected with field-level errors and the circuit was NOT run')

In [ ]:
# ---- Failure-isolation test: an injected failure must not damage earlier completed results ----
def _sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
protected = (sorted(QML_DIR.glob('run_*_seed*.json')) + sorted(PRED_DIR.glob('qml_*_seed*.csv')) + [QML_DIR / 'summary_main.json', SHARED_PATH] +
             [QML_DIR / f for f in ['weights_main_seed42.json', 'preprocessing_main_seed42.joblib', 'history_main_seed42.json']])
before_hash = {str(p): _sha(p) for p in protected}
assert all(json.loads(p.read_text()).get('status') in ('completed', 'failed') for p in QML_DIR.glob('run_*_seed*.json'))
n_completed_before = sum(json.loads(p.read_text()).get('status') == 'completed' for p in QML_DIR.glob('run_*_seed*.json'))

def _boom(): raise RuntimeError('Injected failure for isolation test')
fail_out = safe_run('main', 42, _boom)                    # same target file as the completed seed-42 run
assert fail_out['status'] == 'failed'
assert before_hash == {str(p): _sha(p) for p in protected}, 'a completed result file was modified by a failed run!'
assert json.loads((QML_DIR / 'run_main_seed42.json').read_text())['status'] == 'completed'

tmp_dir = Path(tempfile.mkdtemp()); safe_run('main', 42, _boom, results_dir=tmp_dir)
rec = json.loads((tmp_dir / 'qml' / 'run_main_seed42.json').read_text())
assert rec['status'] == 'failed' and 'Injected failure' in rec['error'] and 'Traceback' not in rec['error']
shutil.rmtree(tmp_dir)
print(f'[PASSED] failure isolation: {len(protected)} protected files unchanged ({n_completed_before} completed runs intact); failed run recorded readably when no earlier result exists')

In [ ]:
# ---- Final summary (re-read from saved files) ----
S = json.loads((QML_DIR / 'summary_main.json').read_text()); r42 = json.loads((QML_DIR / 'run_main_seed42.json').read_text())
msd = lambda a: 'n/a' if a['mean'] is None else (f"{a['mean']:.3f} +/- {a['sd']:.3f}" if a['sd'] is not None else f"{a['mean']:.3f} (n=1)")
print('=' * 100); print('QUANTUM ML (PennyLane variational circuit, SIMULATOR) - FINAL SUMMARY   [family: qml]'); print('=' * 100)
print(f"Backend: {S['backend']} | qubits: {S['n_qubits']} | trainable parameters: {S['n_parameters']} | seeds: {S['seeds']}")
print(f"Config: {json.dumps(S['frozen_config'])}")
print('\nHEADLINE (main, shared reduced feature set) - mean +/- SD over seeds (seed variation only), TEST, unweighted:')
for k in ['sensitivity', 'specificity', 'roc_auc', 'accuracy']: print(f"  {k:<12} @val-chosen thr: {msd(S['metrics_at_threshold'][k])}    @0.5: {msd(S['metrics_at_0.5'][k])}")
m = r42['metrics_at_threshold']; ci = r42['ci95']
print(f"\nSeed-42 run: thr={r42['threshold']:.4f} sensitivity={fmt(m['sensitivity'])} " + ('' if ci['sensitivity'] is None else f"[{ci['sensitivity'][0]:.3f}, {ci['sensitivity'][1]:.3f}] ") +
      f"specificity={fmt(m['specificity'])} ROC-AUC={fmt(m['roc_auc'])} (test n={r42['test_n']}, prevalence={r42['test_prevalence']:.3f}, training subsample {r42['config']['subsample_size']} rows)")
if S['seeds_stopped_early_by_time_budget']: print('Seeds stopped early by the time budget:', S['seeds_stopped_early_by_time_budget'])
print('\nFiles: results/qml/run_main_seed42..46.json | summary_main.json | weights_main_seed42.json | preprocessing_main_seed42.joblib | circuit_diagram.txt')
print('\nNotes: cross-sectional NHANES 2017-2018 classification, NOT prediction of future disease; metrics are UNWEIGHTED; the test set was used once per final model.')
print('Simulator only; no quantum advantage claimed; not comparable to a hardware run. No claim is made about which model family performs best.')
print(WARNING_TEXT)

## Limitations
> ⚠️ **Research/educational use only. Not a diagnosis or medical advice.**

- **Cross-sectional data:** results describe classification within the NHANES 2017–2018 cycle. They are **not** prediction of future disease, and the "Elevated Risk" label is defined only by an HbA1c range (5.7 % to < 6.5 %).
- **Unweighted metrics:** survey weights and design variables were removed, so results do **not** represent the US population and ignore the complex survey design.
- **Simulator only, ≤ 8 qubits, trained on a subsample:** exact classical simulation of a small circuit; no hardware noise or shot noise; the model sees only the shared reduced feature set and a stratified subsample of the training rows (default 1500). **No quantum advantage is claimed**, and results are not comparable to a real-hardware run.
- **Untuned, small model:** 2 entangling layers, one measured qubit, learning rate fixed at 0.05, 15 epochs, no early stopping, classical read-out `sigmoid(a·z + b)`. Different architectures or training settings could change results.
- **Seed SD ≠ uncertainty about generalisation:** 5 seeds share one fixed split and test set; the SD covers subsample draw, initialisation and batch order only. Bootstrap CIs in each run file capture test-set sampling noise and are wide for a test set of this size. No claim of clinical or model-family superiority is made.
- **Time budget:** a seed that exceeds the budget is recorded as failed (default) or labelled as stopped early; stopped-early runs are under-trained and are flagged in the summary.
- **Threshold choice:** max sensitivity with specificity ≥ 0.70 fixed on the full validation set; not clinically validated, and achieved test specificity can fall below 0.70. Positive predictive value depends strongly on prevalence.
- **Fairness not assessed:** sex and race/ethnicity may be predictors; subgroup performance was not evaluated. The score is a research output, not a diagnosis.
- Prediction CSVs under `results/predictions/` are private working files and must not be included in exported reports.